# G11.1 - Multiscale ingestion and EDA

Build the master grid, fault geometries / targets, the DEM catalog, 100 m geophysics features, 1 m DEM summaries aggregated EXACTLY to 100 m cells (2 m work resolution), multimodal alignment, then the spatially rigorous EDA (block-sampled, block-bootstrap CIs).

Nothing in this notebook trains on test labels, uploads to DrivenData or modifies source data. Every stage is resumable: re-running a cell reuses a valid stage record (`manifests/<stage>.json`).

In [ ]:
# Configuration overrides (everything else: configs/g11_multiscale.yaml). Paths are EXPECTED, verified by the audit.
OVERRIDES = {
    "data_root": "/content/drive/MyDrive/GEMS_2",
    "output_root": "outputs/G11_MULTISCALE_FAULT_DISCOVERY",
    "scratch_root": "/content/g11_scratch",
    "reuse_valid_cache": True,
}

In [ ]:
# Mount Drive, pin the source, install, load the config (no data is copied).
from google.colab import drive
drive.mount("/content/drive")
import subprocess, sys
from pathlib import Path
REPO, REF = "mybenkhadda/GEMS---drivendata", "main"
CANDIDATES = ["/content/GEMS", "/content/GEMS---drivendata", "/content/GEMS_G11"]
CODE = next((c for c in CANDIDATES if (Path(c) / ".git").exists()), CANDIDATES[0])
if not (Path(CODE) / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", CODE], check=True)
subprocess.run(["git", "-C", CODE, "fetch", "--depth", "1", "origin", REF], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "--detach", "FETCH_HEAD"], check=True)
if not (Path(CODE) / "gems_bench/g11/pipeline.py").is_file():
    raise RuntimeError(f"{REPO}@{REF} has no gems_bench/g11 - push the G11 package or set REF")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE + "[colab]", "xgboost", "lightgbm"], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
COMMIT = subprocess.check_output(["git", "-C", CODE, "rev-parse", "HEAD"], text=True).strip()
print("checkout", CODE, "commit", COMMIT)

from gems_bench.g11.config import load_config, Mode
from gems_bench.g11.pipeline import G11Pipeline
CFG = load_config(overrides=OVERRIDES)
P = G11Pipeline(CFG)
print("outputs:", P.ctx.root, "| config hash", P.ctx.cfg_hash)
print("Recommended entry point: GEMS_G11_MASTER_COLAB_EXECUTION.ipynb (this notebook runs one phase only)")

In [ ]:
P.audit()  # cheap re-validation of the sources
print(P.ingest())

## Features

Geophysics at 100 m - 3.2 km; DEM aggregation streams only DEM-covered chunks (`dem.aggregate_chunk_cells`); peak RAM is bounded by one chunk + ~15 full-grid arrays.

In [ ]:
print(P.features())
display(P.store.reg.groupby("family").size())

## EDA

Coverage, distributions by label state, near-fault vs local-control contrast with block-bootstrap CIs, utility by scale and region, redundancy, spatial autocorrelation, failure-mode taxonomy and the candidate atlas (1 m hillshade / slope / curvature / cross-fault profiles with tile provenance).

In [ ]:
print(P.eda())
d = P.ctx.d("eda")
from IPython.display import Markdown, Image, display
display(Markdown((d / "eda_summary.md").read_text()))
for f in ["fig_coverage.png", "fig_contrast.png", "fig_utility.png"]:
    if (d / f).is_file(): display(Image(str(d / f)))
print("HTML report:", d / "eda_report.html")